# Clean Halcyon Gas Power Plant Tracker (Aug 2026)

Splits the raw Halcyon export into two CSVs — one for CCGT, one for CT — matching the structure of the earlier `Halcyon_July_CCGT.csv` / `Halcyon_July_CT.csv`, so the existing `CCGT_gas_capex.ipynb` / `CT_gas_capex.ipynb` notebooks can be pointed at this refreshed data.

**Source**: `Halcyon Gas Power Plant Tracker - 24 Aug 2026 .xlsx`, sheet `Gas Plants`.

**Output files**:
- `Halcyon_August_CCGT.csv` — Technology Type = Combined-Cycle Gas Turbine
- `Halcyon_August_CT.csv` — Technology Type = Simple-Cycle Gas Turbine

Both are filtered to rows with a reported `Cost ($/kW)` (the raw export uses `"-"` as a placeholder for missing values).

## 1. Load the raw Excel export

In [1]:
import pandas as pd

RAW_XLSX = "Halcyon Gas Power Plant Tracker - 24 Aug 2026 .xlsx"

raw = pd.read_excel(RAW_XLSX, sheet_name="Gas Plants")
print(raw.shape)
raw["Technology Type"].value_counts()

(473, 78)


Technology Type
Simple-Cycle Gas Turbine                    152
Combined-Cycle Gas Turbine                  127
Reciprocating Internal Combustion Engine    122
Undisclosed / Other                          28
Coal to Gas Conversion                       22
SCGT & CCGT                                   5
-                                             5
SCGT & RICE                                   4
Gas Steam Turbine                             3
RICE & Linear Gen                             2
Conversion                                    1
Oil to Gas Conversion                         1
Repowering Conversion                         1
Name: count, dtype: int64

## 2. Keep the same columns as the July CSVs, and only rows with a reported cost

In [2]:
COLUMNS = [
    "Plant / Unit Name",
    "State",
    "BA Code",
    "Announcement Date",
    "Planned / Operating Year",
    "Technology Type",
    "Capacity (MW)",
    "Cost ($/kW)",
    "CapEx ($M)",
    "Actual vs. Estimated Cost",
    "Dollar Year",
    "Latitude",
    "Longitude",
    "EIA Status",
    "EIA Plant ID",
]

df = raw[COLUMNS].copy()

# Only rows with a reported cost matter for this notebook (regardless of technology type) —
# filter now so the dollar-year normalization below only has to reason about relevant rows.
has_cost = pd.to_numeric(df["Cost ($/kW)"], errors="coerce").notna()
df = df[has_cost].reset_index(drop=True)

# Keep the Announcement Date's year as a plain number for the dollar-year fallback below,
# before the "Announcement Date" column itself is reformatted to the July CSVs' string style.
df["_announcement_year"] = df["Announcement Date"].dt.year

# Match the "Mon D, YYYY" style used in the July CSVs (missing dates -> "-", same placeholder as the rest of the sheet)
df["Announcement Date"] = df["Announcement Date"].dt.strftime("%b %-d, %Y").fillna("-")

print(f"{len(df)} rows with a reported cost, across all technology types")
df.head()

96 rows with a reported cost, across all technology types


,Plant / Unit Name,State,BA Code,Announcement Date,Planned / Operating Year,Technology Type,Capacity (MW),Cost ($/kW),CapEx ($M),Actual vs. Estimated Cost,Dollar Year,Latitude,Longitude,EIA Status,EIA Plant ID,_announcement_year
0,Nikiski Combined Cycle (NCC) Repower Project,AK,-,-,2028,Combined-Cycle Gas Turbine,8,9625,77,Estimated,2026,60.676084,-151.38018,-,-,NaN
1,Thomas Fitzhugh,AR,SWPP,"Feb 26, 2024",-,Simple-Cycle Gas Turbine,140.5,978.947368,93,Estimated,-,35.462382,-93.80493,"Construction complete, but not yet in commerci...",201,2024.0
2,Ironwood Power Station,AR,MISO,"Nov 1, 2024",2028,Simple-Cycle Gas Turbine,446,1753,782,Estimated,-,34.433826,-92.9042,Regulatory approvals pending. Not under constr...,69750,2024.0
3,Independence Gas Plant Units 3-6 (IGP 3-6),AR,MISO,"Nov 3, 2025",2030,Combined-Cycle Gas Turbine,1499,1741.16,2610,Estimated,-,35.679063,-91.407576,-,-,2025.0
4,Jefferson Power Station,AR,MISO,"Aug 1, 2025",2029,Combined-Cycle Gas Turbine,754,2149,1574,-,-,34.420068,-92.159813,Regulatory approvals pending. Not under constr...,69185,2025.0


## 3. Normalize cost to a common dollar year (2022$)

`Cost ($/kW)` values aren't all in the same dollar year (tracker's `Methodology`: *"reference year of reported capital cost values... defaults to the publication date of the filing"* if not stated). We normalize to **2022$** to match `inputs/plant_characteristics/dollaryear.csv` (`gas-ccgt_CEPM_*`, inherited from `gas_ATB_2024_moderate`).

**Method**: chain annual rates from `inputs/financials/inflation_default.csv` (1914-2200, flat 2.5%/yr from 2026) — same source `reeds/financials.py` uses. `deflator.csv` isn't used here since it stops at 2025.

- **$X < 2022$** → **multiply** by rates $X{+}1 \to 2022$ (older dollars are nominally smaller, grow them up).
- **$X > 2022$** → **divide** by rates $2023 \to X$ (later dollars are inflated, shrink them down).

*Example*: Fort Churchill Addition 3, \$2323.04/kW in 2032\$ → factor = 1/(rate(2023)×...×rate(2032)) ≈ 0.7655 → **≈\$1778/kW in 2022\$**.

**Dollar Year used**, in order: (1) explicit `Dollar Year`, (2) `Announcement Date` year as fallback, (3) if both missing, row is **dropped** (2 CCGT rows).

In [3]:
TARGET_DOLLAR_YEAR = 2022

inflation_rate = pd.read_csv(
    "../../../inputs/financials/inflation_default.csv", index_col="t"
)["inflation_rate"]


def dollar_year_factor(from_year, to_year):
    """Multiplier to convert a value in `from_year` dollars to `to_year` dollars,
    by chaining annual inflation rates (same approach as reeds/financials.py)."""
    if from_year == to_year:
        return 1.0
    if from_year < to_year:
        return inflation_rate.loc[from_year + 1 : to_year].prod()
    return 1 / inflation_rate.loc[to_year + 1 : from_year].prod()


# 1. Explicit Dollar Year when given
dollar_year = pd.to_numeric(df["Dollar Year"], errors="coerce")

# 2. Fall back to the Announcement Date's year (the tracker's own documented default)
used_announcement_fallback = dollar_year.isna() & df["_announcement_year"].notna()
dollar_year = dollar_year.fillna(df["_announcement_year"])

# 3. If both are missing, there's no basis to normalize on -- drop these rows
no_basis = dollar_year.isna()
if no_basis.any():
    print(f"Dropping {no_basis.sum()} row(s) with no Dollar Year or Announcement Date:")
    print(df.loc[no_basis, "Plant / Unit Name"].to_string(index=False))
df = df[~no_basis].reset_index(drop=True)
dollar_year = dollar_year[~no_basis].astype(int).reset_index(drop=True)

df["Dollar Year Used"] = dollar_year

raw_cost = pd.to_numeric(df["Cost ($/kW)"], errors="coerce")
factor = dollar_year.apply(lambda y: dollar_year_factor(y, TARGET_DOLLAR_YEAR))
df["Cost ($/kW, 2022$)"] = (raw_cost * factor).round(2)

df = df.drop(columns="_announcement_year")

print(f"\nUsed Announcement Date fallback for {used_announcement_fallback.sum()} row(s).")
df[["Plant / Unit Name", "Dollar Year", "Dollar Year Used", "Cost ($/kW)", "Cost ($/kW, 2022$)"]].head(10)

Dropping 2 row(s) with no Dollar Year or Announcement Date:
Smarr Combined Cycle Energy Facility
      Westlake Power Station CA1/CT1

Used Announcement Date fallback for 27 row(s).


,Plant / Unit Name,Dollar Year,Dollar Year Used,Cost ($/kW),"Cost ($/kW, 2022$)"
0,Nikiski Combined Cycle (NCC) Repower Project,2026,2026,9625,8544.04
1,Thomas Fitzhugh,-,2024,978.947368,913.89
2,Ironwood Power Station,-,2024,1753,1636.50
3,Independence Gas Plant Units 3-6 (IGP 3-6),-,2025,1741.16,1584.26
4,Jefferson Power Station,-,2025,2149,1955.34
5,Springerville Generating Station Repowering Pr...,-,2026,213,189.08
6,Apache Generating Station GT5 and GT6,2025,2025,920.37,837.43
7,Redhawk Plant Expansion,2024,2024,1115.87,1041.71
8,Black Mountain Expansion Project,-,2024,1090,1017.56
9,Sunridge Power Plant,2026,2026,2941,2610.70


## 4. Split into CCGT / CT

In [4]:
ccgt = df[df["Technology Type"] == "Combined-Cycle Gas Turbine"].reset_index(drop=True)
ct = df[df["Technology Type"] == "Simple-Cycle Gas Turbine"].reset_index(drop=True)

print(f"CCGT: {len(ccgt)} plants with reported cost")
print(f"CT:   {len(ct)} plants with reported cost")

CCGT: 32 plants with reported cost
CT:   39 plants with reported cost


## 5. Write the CSVs

In [5]:
ccgt.to_csv("Halcyon_August_CCGT.csv", index=False)
ct.to_csv("Halcyon_August_CT.csv", index=False)

print("Wrote Halcyon_August_CCGT.csv and Halcyon_August_CT.csv")

Wrote Halcyon_August_CCGT.csv and Halcyon_August_CT.csv
